# 🔍 Fake Job Posting Detector
### Complete Pipeline — EDA + Feature Engineering + Modelling

**Dataset:** EMSCAD – Real or Fake Job Postings (Kaggle)  
**Target:** `fraudulent` — 0 = Real, 1 = Fake  
**Run this notebook top to bottom in one session.**


## Step 1 — Install & Download Dataset

In [ ]:
# Install dependencies
!pip install imbalanced-learn xgboost -q

# Download dataset via Kaggle API
import os
os.environ['KAGGLE_USERNAME'] = 'your_kaggle_username'  # replace
os.environ['KAGGLE_KEY']      = 'your_kaggle_api_key'   # replace

!kaggle datasets download -d shivamb/real-or-fake-fake-jobposting-prediction --unzip -p /content/data/

print(os.listdir('/content/data/'))

## Step 2 — Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score
)
from scipy.sparse import hstack, csr_matrix
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from xgboost import XGBClassifier

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print('All libraries loaded.')

## Step 3 — Load Data

In [ ]:
df = pd.read_csv('/content/data/fake_job_postings.csv')
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head(3)

## Step 4 — Class Distribution

In [ ]:
counts = df['fraudulent'].value_counts()
pct    = df['fraudulent'].value_counts(normalize=True) * 100

print(f'Real (0): {counts[0]:,}  ({pct[0]:.1f}%)')
print(f'Fake (1): {counts[1]:,}  ({pct[1]:.1f}%)')
print(f'Imbalance ratio: {counts[0]/counts[1]:.1f}:1')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(['Real', 'Fake'], [counts[0], counts[1]], color=['#2ecc71', '#e74c3c'], width=0.5)
axes[0].set_title('Job Posting Count by Class')
for i, v in enumerate([counts[0], counts[1]]):
    axes[0].text(i, v + 50, f'{v:,}', ha='center', fontweight='bold')
axes[1].pie([pct[0], pct[1]], labels=['Real', 'Fake'],
            colors=['#2ecc71', '#e74c3c'], autopct='%1.1f%%', startangle=90)
axes[1].set_title('Class Proportion')
plt.tight_layout()
plt.show()

## Step 5 — Missing Values

In [ ]:
missing     = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_df  = pd.DataFrame({'count': missing, 'pct': missing_pct})
missing_df  = missing_df[missing_df['count'] > 0].sort_values('pct', ascending=False)
print(missing_df)

plt.figure(figsize=(10, 4))
plt.barh(missing_df.index, missing_df['pct'], color='#e67e22')
plt.xlabel('Missing %')
plt.title('Missing Values by Column')
plt.tight_layout()
plt.show()

## Step 6 — Missing Values by Class (Are Nulls Informative?)

In [ ]:
check_cols = ['company_profile', 'description', 'requirements', 'benefits', 'salary_range']
null_by_class = {}
for col in check_cols:
    if col in df.columns:
        null_by_class[col] = {
            'Real (0)': df[df['fraudulent']==0][col].isnull().mean() * 100,
            'Fake (1)': df[df['fraudulent']==1][col].isnull().mean() * 100
        }
null_class_df = pd.DataFrame(null_by_class).T
print('% Missing by class:')
print(null_class_df.round(1))

null_class_df.plot(kind='bar', color=['#2ecc71','#e74c3c'], figsize=(10,4))
plt.title('Missing % per Column by Class')
plt.ylabel('Missing %')
plt.xticks(rotation=30)
plt.legend(['Real', 'Fake'])
plt.tight_layout()
plt.show()

## Step 7 — Text Length Analysis

In [ ]:
text_cols = ['title', 'description', 'company_profile', 'requirements', 'benefits']
for col in text_cols:
    if col in df.columns:
        df[f'{col}_len'] = df[col].fillna('').apply(len)

len_cols = [c for c in df.columns if c.endswith('_len')]
fig, axes = plt.subplots(1, len(len_cols), figsize=(18, 4))
for ax, col in zip(axes, len_cols):
    real = df[df['fraudulent']==0][col]
    fake = df[df['fraudulent']==1][col]
    ax.hist(real, bins=40, alpha=0.6, color='#2ecc71', label='Real', density=True)
    ax.hist(fake, bins=40, alpha=0.6, color='#e74c3c', label='Fake', density=True)
    ax.set_title(col.replace('_len','').replace('_',' ').title())
    ax.legend(fontsize=8)
plt.suptitle('Text Length: Real vs Fake')
plt.tight_layout()
plt.show()

## Step 8 — Feature Engineering

In [ ]:
df_fe = df.copy()

# Fill text nulls
for col in ['title','company_profile','description','requirements','benefits']:
    df_fe[col] = df_fe[col].fillna('')

# Combined text corpus
df_fe['text'] = (df_fe['title'] + ' ' + df_fe['company_profile'] + ' ' +
                 df_fe['description'] + ' ' + df_fe['requirements'] + ' ' + df_fe['benefits'])

# Missingness flags
df_fe['missing_salary']          = df['salary_range'].isnull().astype(int)
df_fe['missing_company_profile'] = df['company_profile'].isnull().astype(int)
df_fe['missing_requirements']    = df['requirements'].isnull().astype(int)
df_fe['missing_benefits']        = df['benefits'].isnull().astype(int)
df_fe['missing_location']        = df['location'].isnull().astype(int)

# Text length features
df_fe['desc_len']  = df_fe['description'].apply(len)
df_fe['title_len'] = df_fe['title'].apply(len)
df_fe['text_len']  = df_fe['text'].apply(len)

# Red flag keyword count
RED_FLAGS = [
    'urgent', 'immediate', 'no experience', 'work from home',
    'earn money', 'guaranteed', 'wire transfer', 'send personal details',
    'investment required', 'multi-level', 'unlimited earning',
    'be your own boss', 'no degree required'
]
df_fe['red_flag_count'] = df_fe['text'].apply(
    lambda t: sum(1 for f in RED_FLAGS if f in t.lower()))

# Encode categoricals
le = LabelEncoder()
for col in ['employment_type','required_experience','required_education','industry','function']:
    if col in df_fe.columns:
        df_fe[col] = df_fe[col].fillna('Unknown')
        df_fe[col+'_enc'] = le.fit_transform(df_fe[col])

print('Feature engineering done.')
print(f"Red flag count — Fake mean: {df_fe[df_fe['fraudulent']==1]['red_flag_count'].mean():.2f}")
print(f"Red flag count — Real mean: {df_fe[df_fe['fraudulent']==0]['red_flag_count'].mean():.2f}")

## Step 9 — Define Features & Split

In [ ]:
STRUCTURED_FEATURES = [f for f in [
    'telecommuting', 'has_company_logo', 'has_questions',
    'missing_salary', 'missing_company_profile', 'missing_requirements',
    'missing_benefits', 'missing_location',
    'desc_len', 'title_len', 'text_len', 'red_flag_count',
    'employment_type_enc', 'required_experience_enc',
    'required_education_enc', 'industry_enc', 'function_enc'
] if f in df_fe.columns]

X_text   = df_fe['text']
X_struct = df_fe[STRUCTURED_FEATURES]
y        = df_fe['fraudulent']

X_text_train, X_text_test, X_struct_train, X_struct_test, y_train, y_test = train_test_split(
    X_text, X_struct, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {len(y_train):,}  |  Test: {len(y_test):,}')
print(f'Train fraud rate: {y_train.mean():.3f}')

## Step 10 — TF-IDF Vectorisation

In [ ]:
tfidf = TfidfVectorizer(
    max_features=5000, ngram_range=(1, 2),
    stop_words='english', sublinear_tf=True
)

X_tfidf_train = tfidf.fit_transform(X_text_train)
X_tfidf_test  = tfidf.transform(X_text_test)

X_train_final = hstack([X_tfidf_train, csr_matrix(X_struct_train.values)])
X_test_final  = hstack([X_tfidf_test,  csr_matrix(X_struct_test.values)])

print(f'Training matrix : {X_train_final.shape}')
print(f'Test matrix     : {X_test_final.shape}')

## Step 11 — Helper Functions

In [ ]:
def evaluate_model(name, model, X_tr, y_tr, X_te, y_te):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1]
    report = classification_report(y_te, y_pred, output_dict=True)
    auc    = roc_auc_score(y_te, y_prob)
    ap     = average_precision_score(y_te, y_prob)
    f1     = report['1']['f1-score']
    recall = report['1']['recall']
    prec   = report['1']['precision']
    print(f'\n── {name}')
    print(f'  Precision: {prec:.3f}  Recall: {recall:.3f}  F1: {f1:.3f}')
    print(f'  ROC-AUC: {auc:.4f}  PR-AUC: {ap:.4f}')
    return {'name':name,'model':model,'y_pred':y_pred,'y_prob':y_prob,
            'f1':f1,'recall':recall,'precision':prec,'auc':auc,'ap':ap}

def plot_confusion(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Real','Fake'], yticklabels=['Real','Fake'])
    plt.title(title); plt.ylabel('Actual'); plt.xlabel('Predicted')
    plt.tight_layout(); plt.show()

results = {}

## Step 12 — Logistic Regression + SMOTE

In [ ]:
lr_pipeline = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('clf',   LogisticRegression(max_iter=1000, C=1.0, class_weight='balanced', random_state=42))
])
results['lr'] = evaluate_model('Logistic Regression + SMOTE',
                                lr_pipeline, X_train_final, y_train, X_test_final, y_test)
plot_confusion(y_test, results['lr']['y_pred'], 'Logistic Regression')

## Step 13 — Random Forest + SMOTE

In [ ]:
rf_pipeline = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('clf',   RandomForestClassifier(n_estimators=200, max_depth=20,
                                      class_weight='balanced', random_state=42, n_jobs=-1))
])
results['rf'] = evaluate_model('Random Forest + SMOTE',
                                rf_pipeline, X_train_final, y_train, X_test_final, y_test)
plot_confusion(y_test, results['rf']['y_pred'], 'Random Forest')

## Step 14 — XGBoost + SMOTE

In [ ]:
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
spw = neg / pos
print(f'scale_pos_weight = {spw:.1f}')

xgb_pipeline = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('clf',   XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05,
                             subsample=0.8, colsample_bytree=0.8,
                             scale_pos_weight=spw, eval_metric='logloss',
                             random_state=42, n_jobs=-1))
])
results['xgb'] = evaluate_model('XGBoost + SMOTE',
                                  xgb_pipeline, X_train_final, y_train, X_test_final, y_test)
plot_confusion(y_test, results['xgb']['y_pred'], 'XGBoost')

## Step 15 — Model Comparison

In [ ]:
comparison = pd.DataFrame([
    {'Model': r['name'], 'Fake Precision': r['precision'],
     'Fake Recall': r['recall'], 'Fake F1': r['f1'],
     'ROC-AUC': r['auc'], 'PR-AUC': r['ap']}
    for r in results.values()
]).set_index('Model')

print(comparison.round(4))

comparison[['Fake Recall','Fake F1','ROC-AUC']].plot(
    kind='bar', figsize=(10,5), color=['#e74c3c','#3498db','#2ecc71'])
plt.title('Model Comparison')
plt.ylabel('Score')
plt.xticks(rotation=20, ha='right')
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

## Step 16 — ROC & PR Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = {'lr': '#9b59b6', 'rf': '#e67e22', 'xgb': '#e74c3c'}

for key, r in results.items():
    fpr, tpr, _ = roc_curve(y_test, r['y_prob'])
    axes[0].plot(fpr, tpr, color=colors[key], label=f"{r['name']} (AUC={r['auc']:.3f})")
    prec, rec, _ = precision_recall_curve(y_test, r['y_prob'])
    axes[1].plot(rec, prec, color=colors[key], label=f"{r['name']} (AP={r['ap']:.3f})")

axes[0].plot([0,1],[0,1],'k--',alpha=0.4)
axes[0].set_title('ROC Curve'); axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].legend(fontsize=9)
axes[1].set_title('Precision-Recall Curve')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].legend(fontsize=9)
plt.tight_layout()
plt.show()

## Step 17 — Hyperparameter Tuning (XGBoost)
**This step takes 5–10 minutes. Let it run.**

In [ ]:
param_dist = {
    'clf__n_estimators':     [200, 300, 500],
    'clf__max_depth':        [4, 6, 8],
    'clf__learning_rate':    [0.01, 0.05, 0.1],
    'clf__subsample':        [0.7, 0.8, 1.0],
    'clf__colsample_bytree': [0.7, 0.8, 1.0],
    'clf__min_child_weight': [1, 3, 5],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

search = RandomizedSearchCV(
    xgb_pipeline, param_distributions=param_dist,
    n_iter=20, scoring='f1', cv=cv,
    verbose=1, n_jobs=-1, random_state=42
)

search.fit(X_train_final, y_train)
print(f'\nBest CV F1 : {search.best_score_:.4f}')
print(f'Best params: {search.best_params_}')

## Step 18 — Final Evaluation

In [ ]:
best_model = search.best_estimator_

y_pred_final = best_model.predict(X_test_final)
y_prob_final = best_model.predict_proba(X_test_final)[:, 1]

print('Classification Report — Tuned XGBoost:')
print(classification_report(y_test, y_pred_final, target_names=['Real','Fake']))
print(f'ROC-AUC: {roc_auc_score(y_test, y_prob_final):.4f}')

plot_confusion(y_test, y_pred_final, 'Tuned XGBoost — Final')

## Step 19 — Save Model Files

In [ ]:
import os
os.makedirs('/content/model', exist_ok=True)

joblib.dump(best_model,          '/content/model/best_model.pkl')
joblib.dump(tfidf,               '/content/model/tfidf_vectorizer.pkl')
joblib.dump(STRUCTURED_FEATURES, '/content/model/structured_features.pkl')

print('Saved:', os.listdir('/content/model/'))

## Step 20 — Download Model Files to Your PC

In [ ]:
from google.colab import files

files.download('/content/model/best_model.pkl')
files.download('/content/model/tfidf_vectorizer.pkl')
files.download('/content/model/structured_features.pkl')

print('Downloaded all 3 model files to your PC.')
print('Place them in the model/ folder of your project before running the Streamlit app.')